# Step 1: Choose plates to analyze [PLATE_DIRS] and output location for analysis files [OUTPUT_DIR]

In [ ]:
#Cell 1 — Imports & config
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import sys
import importlib
import utils.visualization
from utils.quiver_style import QUIVER_NEURON_COLORS
from utils.scs_utils import compute_psp_props_fov
from utils.visualization import (
    build_condition_feature_df, filter_fovs,
    build_variance_table, _cohens_d,
    plot_condition_heatmap, plot_plate_heatmap,
    plot_well_feature_heatmap, plot_plate_condition_heatmap,
    plot_network_fingerprint, plot_circuit_composition,
    plot_fanin_fanout, plot_feature_boxplots, pool_conditions,
    plot_pharmacology_dissociation, plot_synaptic_coupling,
)

# Choose folder to save this notebook's output
from datetime import date
OUTPUT_DIR = os.path.join(r'R:\QNP\2026_QNP\QNP066_068-PlateComparisons\DLX_selection_protocol',
date.today().strftime('%Y-%m-%d'))
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Point each entry to a plate's OUTPUT_DIR
PLATE_DIRS = {
    #'QNP-066_Plate2': r'R:\QNP\2026_QNP\2026-05-08_JenniferGrooms_FireflyOne\PlateNumber_2\v16_outputs_20260514_1516', # No Drugs
    #'QNP-066_Plate3': r'R:\QNP\2026_QNP\2026-05-08_JenniferGrooms_FireflyOne\PlateNumber_3\v16_outputs_20260514_1556', # No Drugs
    #'QNP-068_Plate1': r'R:\QNP\2026_QNP\2026-05-12_JenniferGrooms_FireflyOne\PlateNumber_1\v16_outputs_20260519_1541', # GABAzine
    #'QNP-068_Plate2': r'R:\QNP\2026_QNP\2026-05-12_JenniferGrooms_FireflyOne\PlateNumber_2\v16_outputs_20260514_1241', # NBQX
    #'QNP-068_Plate3': r'R:\QNP\2026_QNP\2026-05-13_JenniferGrooms_FireflyOne\PlateNumber_3\v16_outputs_20260515_0916', # Drugs
    #'QNP-068_Plate4': r'R:\QNP\2026_QNP\2026-05-13_JenniferGrooms_FireflyOne\PlateNumber_4\v16_outputs_20260515_1255', # Drugs
    #'QNP-069_Plate1': r'R:\QNP\2026_QNP\2026-05-19_JenniferGrooms_FireflyOne\PlateNumber_1\v16_outputs_20260520_1205', # new DLX protocol
    #'QNP-069_Plate2': r'R:\QNP\2026_QNP\2026-05-19_JenniferGrooms_FireflyOne\PlateNumber_2_forGrace\v16_outputs_20260521_1551', # new DLX protocol
    #'QNP-069_Plate3': r'R:\QNP\2026_QNP\2026-05-20_JenniferGrooms_FireflyOne\PlateNumber_3_forGrace\v16_outputs_20260521_1729', # new DLX protocol
    #'QNP-069_Plate4': r'R:\QNP\2026_QNP\2026-05-20_JenniferGrooms_FireflyOne\PlateNumber_4\v16_outputs_20260522_0942', # new DLX protocol

}

CONTROL_LABELS = {'No Drug', 'NoDrug', 'DMSO', 'dmso', ''}

# Step 2: Load previously analyzed data from pickle files. Approximately 2 minutes to load per plate

In [ ]:
# Cell 2 — Load data from each plate
all_fov_results = []
df_conn_list    = []
df_neuron_list  = []
df_dlx_list     = []

for plate_label, out_dir in PLATE_DIRS.items():
    # pickle
    pkl = os.path.join(out_dir, 'all_fov_results_cache.pkl')
    if os.path.exists(pkl):
        with open(pkl, 'rb') as f:
            plate_results = pickle.load(f)
        for R in plate_results:
            R['_plate'] = plate_label
        all_fov_results.extend(plate_results)
    else:
        print(f'[{plate_label}] no pickle found — skipping FOV-level analyses')

    # CSVs
    for fname, lst in (
        ('merged_connections_all_fovs.csv', df_conn_list),
        ('merged_neurons_all_fovs.csv',     df_neuron_list),
    ):
        path = os.path.join(out_dir, fname)
        if os.path.exists(path):
            df = pd.read_csv(path)
            df['plate'] = plate_label
            lst.append(df)
        else:
            print(f'[{plate_label}] {fname} not found')

    dlx_path = os.path.join(out_dir, 'dlx_accuracy_per_fov.csv')
    if os.path.exists(dlx_path):
        df = pd.read_csv(dlx_path)
        df['plate'] = plate_label
        df_dlx_list.append(df)
fov_well_map      = {}
fov_condition_map = {}
for plate_label, out_dir in PLATE_DIRS.items():
    path = os.path.join(out_dir,
'merged_connections_all_fovs_with_metadata.csv')
    if os.path.exists(path):
        available = pd.read_csv(path, nrows=0).columns.tolist()
        cols = [c for c in ('fov', 'wellId', 'drug1') if c in available]
        md = pd.read_csv(path, usecols=cols)
        for fov, well in md.groupby('fov')['wellId'].first().items():
            fov_well_map[(fov, plate_label)] = well
        if 'drug1' in md.columns:
            for fov, cond in md.groupby('fov')['drug1'].first().items():
                fov_condition_map[(fov, plate_label)] = str(cond).strip() if pd.notna(cond) else 'No Drug'

df_conn    = pd.concat(df_conn_list,   ignore_index=True) if df_conn_list   else pd.DataFrame()
df_neurons = pd.concat(df_neuron_list, ignore_index=True) if df_neuron_list else pd.DataFrame()
df_dlx     = pd.concat(df_dlx_list,    ignore_index=True) if df_dlx_list    else pd.DataFrame()

print(f'Loaded {len(all_fov_results)} FOVs across {len(PLATE_DIRS)} plates')
print(f'  Connections: {len(df_conn)} rows')
print(f'  Neurons:     {len(df_neurons)} rows')
print(f'  DLX FOVs:    {len(df_dlx)} rows')

In [ ]:
# Cell 3 — Build per-FOV feature table
df_features = build_condition_feature_df(all_fov_results)
df_features['plate'] = [R['_plate'] for R in all_fov_results]
df_features['well'] = [fov_well_map.get((R['fov'], R['_plate'])) for R in
all_fov_results]
df_features['condition'] = [fov_condition_map.get((R['fov'], R['_plate']), 'No Drug') for R in all_fov_results]

if not df_dlx.empty:
    keep = [c for c in ('fov', 'plate', 'pct_inh_correct', 'pct_exc_correct',
'threshold')
            if c in df_dlx.columns]
    df_features = df_features.merge(
        df_dlx[keep].rename(columns={
            'pct_inh_correct': 'dlx_pct_inh_correct',
            'pct_exc_correct': 'dlx_pct_exc_correct',
            'threshold':       'dlx_threshold',
        }),
        on=['fov', 'plate'], how='left',
    )

df_features.head()

In [ ]:
# Cell 4 — Validation rate per plate
if not df_conn.empty:
    val_tiers = {'both', 'scs_only', 'sta_only'}
    df_conn['validated'] = df_conn['consensus_tier'].isin(val_tiers)

    val_rate = (df_conn.groupby(['plate', 'fov'])
                .agg(n_total=('validated', 'size'),
                    n_validated=('validated', 'sum'))
                .assign(pct_validated=lambda x: 100 * x.n_validated / x.n_total)
                .reset_index())
    X_SCALE = 0.3  # controls spacing between bars
    width = 0.2
    fig, ax = plt.subplots(figsize=(len(PLATE_DIRS) * X_SCALE + 1.5, 3))
    for i, (plate, grp) in enumerate(val_rate.groupby('plate')):
        vals = grp['pct_validated']
        n = len(vals)
        mean, sem = vals.mean(), vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0
        x = i * X_SCALE
        ax.bar(x, mean, yerr=sem, color='gray', width=width, capsize=5, error_kw=dict(elinewidth=1.5))
        ax.scatter([x] * n, vals, color='k', s=20, zorder=5, alpha=0.6)
        ax.text(x, 2, f'{n} FOVs', ha='center', va='bottom', fontsize=6)

    ax.set_xticks([i * X_SCALE for i in range(len(PLATE_DIRS))])
    ax.set_xticklabels(list(PLATE_DIRS.keys()), rotation=30, ha='right', fontsize=7)
    ax.set_ylabel('Validated connections (%)')
    ax.set_title('Validation rate per plate')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'validation_rate_per_plate.png'), dpi=300, bbox_inches='tight')
    plt.close()

# Exclude FOVs (optional)

In [ ]:
# # Exclude particular FOVs
#df_plot = df_features[~((df_features['fov'] == 'FOV_0047') & (df_features['plate'] == 'QNP-068_Plate1'))]
# drop later FOVs by number
#df_plot = filter_fovs(df_features, max_fov_per_plate=36)

# explicit exclusions
#df_plot = filter_fovs(df_features, exclude=['fov_0001'])

# keep only first 3 FOVs per well (useful if later acquisitions drift)
#df_plot = filter_fovs(df_features, max_per_well=3)

# combine — filters apply in order: exclude → max_fov → max_per_well
#df_plot = filter_fovs(df_features, max_fov=25, exclude=['fov_007'], max_per_well=4)

In [ ]:
# Cell 5 — Connection counts (EXC / INH) and confidence scores
val_tiers = {'both', 'scs_only', 'sta_only'}
plates = list(PLATE_DIRS.keys())
if not df_conn.empty and 'consensus_tier' in df_conn.columns:
    val = df_conn[df_conn['consensus_tier'].isin(val_tiers)].copy()
    val['type'] = val['type'].str.upper()
    val['score'] = val[['scs_score_p1', 'sta_score_p1']].max(axis=1)

    # Remove outlier FOVs (connection count > Q3 + 3*IQR per plate/type)
    fov_counts = val.groupby(['plate', 'fov', 'type']).size().reset_index(name='n')
    thresholds = (fov_counts.groupby(['plate', 'type'])['n']
                .quantile([0.25, 0.75]).unstack()
                .rename(columns={0.25: 'q1', 0.75: 'q3'})
                .reset_index())
    thresholds['limit'] = thresholds['q3'] + 3 * (thresholds['q3'] - thresholds['q1'])
    fov_counts_clean = (fov_counts
                        .merge(thresholds[['plate', 'type', 'limit']], on=['plate', 'type'])
                        .query('n <= limit')
                        .drop(columns='limit'))
    clean_fovs = fov_counts_clean[['plate', 'fov']].drop_duplicates()
    val_clean = val.merge(clean_fovs, on=['plate', 'fov'])

    fig, axes = plt.subplots(1, 2, figsize=(max(6, len(plates) * 1.8), 3))

    # Connection counts
    counts = (val_clean.groupby(['plate', 'fov', 'type'])
            .size().reset_index(name='n')
            .groupby(['plate', 'type'])['n'].agg(['mean', 'sem'])
            .reset_index())
    
    x = np.arange(len(plates))
    width = 0.3

    for j, typ in enumerate(['EXC', 'INH']):
        color = QUIVER_NEURON_COLORS['Inhibitory'] if label == 'INH' else QUIVER_NEURON_COLORS['Excitatory']
        sub = counts[counts['type'] == typ].set_index('plate').reindex(plates)
        offset = (j - 0.5) * width
        axes[0].bar(x + offset, sub['mean'], yerr=sub['sem'],
                    width=width, color=color, capsize=4, label=typ)

    axes[0].set_xticks(x)
    axes[0].set_xticklabels(plates, ha='center')
    axes[0].set_ylabel('Mean connections per FOV')
    axes[0].set_title('EXC / INH connection counts')
    axes[0].legend(frameon=False)
    axes[0].spines[['top', 'right']].set_visible(False)

    # Confidence scores
    for i, (plate, grp) in enumerate(val_clean.groupby('plate')):
        for j, (typ, color) in enumerate([('EXC', QUIVER_NEURON_COLORS['Excitatory'])], ('INH', QUIVER_NEURON_COLORS['Inhibitory']]):
            scores = grp.loc[grp['type'] == typ, 'score'].dropna()
            axes[1].scatter([i + j * 0.3] * len(scores), scores,
                            color=color, s=10, alpha=0.4)
            if len(scores):
                axes[1].plot([i + j * 0.3], [scores.mean()],
                            'o', color=color, markersize=7, zorder=5)

    axes[1].set_xticks(range(len(PLATE_DIRS)))
    axes[1].set_xticks([i + 0.15 for i in range(len(PLATE_DIRS))])
    axes[1].set_xticklabels(list(PLATE_DIRS.keys()), ha='center', fontsize=8)
    axes[1].set_ylabel('Confidence score')
    axes[1].set_title('Connection confidence scores')
    axes[1].spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'connection_counts_confidence.png'), dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
#Cell 6 — PSP properties per plate
if not df_conn.empty:
    psp_rows = []
    for R in all_fov_results:
        props = compute_psp_props_fov(R)
        if props is None:
            continue
        row = {'fov': R['fov'], 'plate': R['_plate']}
        for typ in ('EXC', 'INH'):
            for k, v in props[typ].items():
                row[f'{k}_{typ.lower()}'] = v
        psp_rows.append(row)
    df_psp = pd.DataFrame(psp_rows)

    _psp_props = [
        ('amplitude',      'Amplitude (ΔF/F)'),
        ('auc',            'AUC (ΔF/F · s)'),
        ('rise_time_ms',   'Rise time (ms)'),
        ('half_width_ms',  'Half-width (ms)'),
        ('decay_time_ms',  'Decay time to 50% (ms)'),
        ('snr',            'SNR'),
    ]

    plates = list(PLATE_DIRS.keys())
    fig, axes = plt.subplots(2, 3, figsize=(8, 8))
    for ax_idx, (ax, (prop, ylabel)) in enumerate(zip(axes.flatten(), _psp_props)):
        for i, plate in enumerate(plates):
            grp = df_psp[df_psp['plate'] == plate]
            for j, (typ, color) in enumerate([('exc', '#5BA85A'), ('inh', '#E07A2F')]):
                vals = grp[f'{prop}_{typ}'].dropna()
                if ax_idx == 5:
                    vals = vals[vals <= 1000]
                n = len(vals)
                mean = vals.mean() if n else 0
                sem  = vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0
                x = i * 2.5 + j * 0.9
                ax.bar(x, mean, yerr=sem, color=color, width=0.7, capsize=4,
                        error_kw=dict(elinewidth=1.2),
                        label=typ.upper() if i == 0 else '')
                ax.scatter([x] * n, vals, color='k', s=12, zorder=5, alpha=0.5)
        ax.set_xticks([i * 2.5 + 0.45 for i in range(len(plates))])
        ax.set_xticklabels(plates, rotation=30, ha='right', fontsize=8)
        ax.set_ylabel(ylabel)
        ax.spines[['top', 'right']].set_visible(False)
        if ax is axes.flatten()[0]:
            ax.legend(frameon=False, fontsize=8)
    plt.suptitle('PSP properties across plates', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'psp_properties.png'), dpi=300,
    bbox_inches='tight')
    plt.close()

In [ ]:
# Cell 7 — DLX accuracy per plate
if not df_dlx.empty:
    fig, axes = plt.subplots(1, 2, figsize=(6, 4))
    for ax, (col, label, color) in zip(axes, [
        ('pct_inh_correct', 'INH correctly DLX+', '#E07A2F'),
        ('pct_exc_correct', 'EXC correctly DLX−', '#5BA85A'),
    ]):
        for i, (plate, grp) in enumerate(df_dlx.groupby('plate')):
            vals = grp[col].dropna()
            n = len(vals)
            mean = vals.mean() if n else 0
            sem  = vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0
            ax.bar(i, mean, yerr=sem, color=color, width=0.6, capsize=5,
                    error_kw=dict(elinewidth=1.5))
            ax.scatter([i] * n, vals, color='k', s=20, zorder=5, alpha=0.6)
            ax.text(i, 2, f'n={n}', ha='center', va='bottom', fontsize=8)
        ax.set_xticks(range(df_dlx['plate'].nunique()))
        ax.set_xticklabels(df_dlx['plate'].unique(), rotation=30, ha='right')
        ax.set_ylabel(f'{label} (%)')
        ax.set_ylim(0, 110)
        ax.set_title(label)
        ax.spines[['top', 'right']].set_visible(False)
    plt.suptitle('DLX cell identity accuracy', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'neuron_composition_DLX.png'), dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
#  Cell 8 — Validation rate per FOV — EXC / INH strip plot across plates
if not df_conn.empty:
    total = (df_conn.groupby(['plate', 'fov', 'type'])
            .size().reset_index(name='total'))
    validated = (df_conn[df_conn['consensus_tier'].isin(val_tiers)]
                .groupby(['plate', 'fov', 'type'])
                .size().reset_index(name='n_val'))
    vr = total.merge(validated, on=['plate', 'fov', 'type'], how='left')
    vr['n_val'] = vr['n_val'].fillna(0)
    vr['rate'] = 100 * vr['n_val'] / vr['total']
    vr['type'] = vr['type'].str.upper()

    plates = list(PLATE_DIRS.keys())
    fig, ax = plt.subplots(figsize=(max(3, len(plates) * 1.2), 3))

    for i, plate in enumerate(plates):
        for j, (typ, color) in enumerate([('EXC', '#5BA85A'), ('INH', '#E07A2F')]):
            rates = vr.loc[(vr['plate'] == plate) & (vr['type'] == typ), 'rate'].dropna()
            x = i * 1.2 + j * 0.5
            x_jit = np.random.uniform(x - 0.2, x + 0.2, size=len(rates))
            ax.scatter(x_jit, rates, color=color, s=18, alpha=0.5,
                        label=typ if i == 0 else '')
            if len(rates):
                ax.plot(x, rates.mean(), 'o', color=color, markersize=8,
                        markeredgecolor='k', markeredgewidth=0.8, zorder=5)

    ax.set_xticks([i * 1.2 + 0.25 for i in range(len(plates))])
    ax.set_xticklabels(plates, rotation=0, ha='center', fontsize=8)
    ax.set_ylabel('Validation rate (%)')
    ax.set_title('Validation rate per FOV across plates')
    #ax.legend(frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'validation_rate_strip.png'), dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
#  Cell 9 — Pharmacological validation
if not df_conn.empty:
    GABAZINE = 'GABAzine'
    CNQX     = 'DAP5-NBQX-CNQX'  # adjust to match your condition label
    VAL_TIERS = {'both', 'scs_only', 'sta_only'}

    # Build per-FOV metrics for all four conditions
    rows = []
    for R in all_fov_results:
        fov   = R['fov']
        cond  = str(fov_condition_map.get((fov, R['_plate']), '') or '').strip() or 'No Drug'
        if cond not in ('No Drug', GABAZINE, CNQX):
            continue

        scs    = R.get('scs1')
        merged = R.get('merged')
        meta   = R.get('meta', {})
        fs     = int(meta.get('fs', 500) or 500)
        rec_s  = (int(meta.get('spont_n_samples', 0) or 0) +
                int(meta.get('stim_n_samples',  0) or 0)) / fs

        epsp_n = sum(len(v) for v in scs.get('epsps_all', {}).values()) if scs else 0
        ipsp_n = sum(len(v) for v in scs.get('ipsps_all', {}).values()) if scs else 0

        if merged is not None and not merged.empty:
            inh = merged[merged['type'].str.upper() == 'INH']
            exc = merged[merged['type'].str.upper() == 'EXC']
            inh_val = inh['consensus_tier'].isin(VAL_TIERS).sum()
            exc_val = exc['consensus_tier'].isin(VAL_TIERS).sum()
            inh_scores = pd.concat([inh[c].dropna() for c in
    ('scs_score_p1','sta_score_p1') if c in inh.columns])
            exc_scores = pd.concat([exc[c].dropna() for c in
    ('scs_score_p1','sta_score_p1') if c in exc.columns])
            inh_score = float(inh_scores.mean()) if len(inh_scores) else np.nan
            exc_score = float(exc_scores.mean()) if len(exc_scores) else np.nan
        else:
            inh_val = exc_val = 0
            inh_score = exc_score = np.nan

        rows.append({'condition': cond,
                    'ipsp_rate': ipsp_n / rec_s if rec_s else np.nan,
                    'epsp_rate': epsp_n / rec_s if rec_s else np.nan,
                    'inh_val':   inh_val,
                    'exc_val':   exc_val,
                    'inh_score': inh_score,
                    'exc_score': exc_score})

    df_pharm = pd.DataFrame(rows)

    # Plot
    fig, axes = plt.subplots(2, 3, figsize=(8, 6))

    panels = [
        # (ax_row, ax_col, metric,      No Drug col,   Drug col,   drug, color,     ylabel)
        (0, 0, 'ipsp_rate',  'No Drug', GABAZINE, '#DD52D1', 'IPSP rate (Hz)'),
        (0, 1, 'inh_val',    'No Drug', GABAZINE, '#DD52D1', 'INH validated connections'),
        (0, 2, 'inh_score',  'No Drug', GABAZINE, '#DD52D1', 'INH confidence score'),
        (1, 0, 'epsp_rate',  'No Drug', CNQX,     "#FD0B70", 'EPSP rate (Hz)'),
        (1, 1, 'exc_val',    'No Drug', CNQX,     '#FD0B70', 'EXC validated connections'),
        (1, 2, 'exc_score',  'No Drug', CNQX,     '#FD0B70', 'EXC confidence score'),
    ]

    for row, col, metric, ctrl_lbl, drug_lbl, drug_color, ylabel in panels:
        ax = axes[row, col]
        conditions = [ctrl_lbl, drug_lbl]
        colors     = ['gray', drug_color]

        ctrl_vals = df_pharm.loc[df_pharm['condition'] == ctrl_lbl,
    metric].dropna()
        drug_vals = df_pharm.loc[df_pharm['condition'] == drug_lbl,
    metric].dropna()
        d = _cohens_d(drug_vals.values, ctrl_vals.values)

        for i, (cond, color) in enumerate(zip([ctrl_lbl, drug_lbl], ['gray', drug_color])):
            x    = i * X_SCALE
            vals = df_pharm.loc[df_pharm['condition'] == cond, metric].dropna()
            n    = len(vals)
            mean = vals.mean() if n else 0
            sem  = vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0
            ax.bar(x, mean, yerr=sem, color=color, width=0.3,
                    capsize=5, error_kw=dict(elinewidth=1.5))
            ax.scatter([x] * n, vals, color='k', s=15, zorder=5, alpha=0.6)
            #ax.text(x, 0, f'n={n} FOV', ha='center', va='bottom', fontsize=6)

        ax.set_xticks([i * X_SCALE for i in range(2)])
        ax.set_xticklabels([ctrl_lbl, drug_lbl], rotation=20, ha='right', fontsize=7)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(f'cohen d={d:.2f}' if not np.isnan(d) else '', fontsize=8)
        ax.spines[['top', 'right']].set_visible(False)
        ax.set_title(f'cohen d={d:.2f}' if not np.isnan(d) else '', fontsize=8)
        if col == 1:
            if row == 1:
                ax.set_ylim(bottom=0, top=500)
            else:
                ax.set_ylim(bottom=0, top=150)
        ax.spines[['top', 'right']].set_visible(False)

    fig.text(0, 0.75, 'GABAzine', va='center', rotation='vertical',
            fontsize=8, fontweight='bold', color='#DD52D1')
    fig.text(0, 0.25, 'CNQX', va='center', rotation='vertical',
            fontsize=8, fontweight='bold', color='#FD0B70')

    plt.suptitle('Pharmacological validation — INH and EXC blockade', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'pharmacological_validation.png'),
    dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# Cell 10 — 
# Stim APs vs evoked partner responses, normalized to chances 
# Shows the increase in evoked activity detection with AP rate (left)
# and probability of detecting a response per AP (right)
if not df_conn.empty:
    VAL_TIERS = {'both', 'scs_only', 'sta_only'}

    neuron_rows = []
    for R in all_fov_results:
        merged = R.get('merged')
        sta    = R.get('sta1', {})
        if merged is None or merged.empty or not sta:
            continue

        val = merged[merged['consensus_tier'].isin(VAL_TIERS)]
        if val.empty:
            continue

        sta_results = sta.get('all_results', {})
        aps_for_sta = sta.get('aps_for_sta', {})

        for pre, grp in val.groupby('pre'):
            n_stim_aps = len(aps_for_sta.get(pre, []))
            if n_stim_aps == 0:
                continue

            corrected_total  = 0
            fractions        = []
            n_valid_partners = 0

            for _, conn in grp.iterrows():
                result = sta_results.get((pre, conn['post']))
                if result is None:
                    continue
                n_psp       = getattr(result, 'n_psp',           0) or 0
                n_ap_evoked = getattr(result, 'n_ap_evoked',     0) or 0
                n_coinc     = getattr(result, 'n_coinc_dropped', 0) or 0
                observable  = n_stim_aps - n_coinc
                if observable <= 0:
                    continue
                corrected_total += (n_psp + n_ap_evoked) * n_stim_aps / observable
                fractions.append((n_psp + n_ap_evoked) / observable)
                n_valid_partners += 1

            if n_valid_partners == 0 or corrected_total == 0:
                continue

            neuron_rows.append({
                'plate':             R['_plate'],
                'n_stim_aps':        n_stim_aps,
                'evoked_per_partner': corrected_total / n_valid_partners,
                'response_fraction': np.mean(fractions),
            })

    df_neuron = pd.DataFrame(neuron_rows)

    def _iqr_mask(s, k=3.0):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        return (s >= q1 - k * iqr) & (s <= q3 + k * iqr)

    mask = (_iqr_mask(df_neuron['n_stim_aps']) &
            _iqr_mask(df_neuron['evoked_per_partner']) &
            _iqr_mask(df_neuron['response_fraction']))
    df_p = df_neuron[mask]
    n_removed = len(df_neuron) - mask.sum()

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))

    for ax, ycol, ylabel, ylim in [
        (axes[0], 'evoked_per_partner', 'Evoked responses per partner\n(coincidence-corrected)',  None),
        (axes[1], 'response_fraction',  'Mean response fraction per partner\n(evoked / observable windows)', (0, 1)),
    ]:
        x = df_p['n_stim_aps'].values
        y = df_p[ycol].values

        ax.scatter(x, y, color='#5B8DB8', s=15, alpha=0.5)

        if len(x) > 2:
            slope, intercept, r, p, _ = stats.linregress(x, y)
            xfit = np.linspace(x.min(), x.max(), 100)
            ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
            ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                    transform=ax.transAxes, fontsize=8)

        ax.set_xlabel('Pre-synaptic APs during stimulation', fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.set_title(f'n={len(df_p)} neurons, {n_removed} removed', fontsize=8)
        ax.set_xlim(left=0)
        ax.set_ylim(bottom=0) if ylim is None else ax.set_ylim(ylim)
        ax.spines[['top', 'right']].set_visible(False)

    axes[0].set_title(f'Count: scales with firing\n(n={len(df_p)} neurons, {n_removed} removed)', fontsize=10)
    axes[1].set_title(f'Fraction: per-spike reliability\n(n={len(df_p)} neurons, {n_removed} removed)', fontsize=10)

    fig.suptitle('Stim APs vs evoked partner responses', fontsize=10)
    fig.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR,
    'stim_aps_vs_evoked_count_and_fraction.png'), dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# Cell 11 — 
if not df_conn.empty:
    fov_rows = []
    for R in all_fov_results:
        scs  = R.get('scs1')
        meta = R.get('meta', {})
        if not scs:
            continue

        fs    = int(meta.get('fs', 500) or 500)
        rec_s = (int(meta.get('spont_n_samples', 0) or 0) +
                int(meta.get('stim_n_samples',  0) or 0)) / fs
        if rec_s == 0:
            continue

        aps_all   = scs.get('aps_all',   {})
        epsps_all = scs.get('epsps_all', {})
        ipsps_all = scs.get('ipsps_all', {})

        active = {k: v for k, v in aps_all.items() if len(v) > 0}
        n_active = len(active)
        if n_active == 0:
            continue

        n_aps  = sum(len(v) for v in active.values())
        n_psps = sum(len(v) for v in epsps_all.values()) + sum(len(v) for v in
    ipsps_all.values())

        fov_rows.append({
            'plate':    R['_plate'],
            'ap_rate':  (n_aps / n_active) / rec_s,
            'psp_rate': n_psps / rec_s,
        })

    df_fov_sc = pd.DataFrame(fov_rows)

    def _iqr_mask(s, k=3):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        return (s >= q1 - k * iqr) & (s <= q3 + k * iqr)

    mask = _iqr_mask(df_fov_sc['ap_rate']) & _iqr_mask(df_fov_sc['psp_rate'])
    df_plot = df_fov_sc[mask]

    fig, ax = plt.subplots(figsize=(4, 3.5))
    x = df_plot['ap_rate'].values
    y = df_plot['psp_rate'].values

    ax.scatter(x, y, color='#5B8DB8', s=20, alpha=0.6)

    if len(x) > 2:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit = np.linspace(x.min(), x.max(), 100)
        ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
        ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                transform=ax.transAxes, fontsize=8)

    n_removed = len(df_fov_sc) - mask.sum()
    ax.set_xlabel('Mean AP rate per active neuron (Hz)', fontsize=8)
    ax.set_ylabel('Total PSP rate (Hz)', fontsize=8)
    ax.set_title(f'PSP rate scales with AP rate (n={len(df_plot)}, {n_removed} removed)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'ap_vs_psp_scaling.png'), dpi=300,
    bbox_inches='tight')
    plt.close()

In [ ]:
# Cell XX — 
# EXC pre AP vs post EPSP rate per FOV
if not df_conn.empty:
    VAL_TIERS = {'both', 'scs_only', 'sta_only'}

    fov_rows = []
    for R in all_fov_results:
        scs        = R.get('scs1')
        merged     = R.get('merged')
        meta       = R.get('meta', {})
        sta_results = R.get('sta1', {}).get('all_results', {})
        if not scs or merged is None or merged.empty:
            continue

        fs    = int(meta.get('fs', 500) or 500)
        rec_s = (int(meta.get('spont_n_samples', 0) or 0) +
                int(meta.get('stim_n_samples',  0) or 0)) / fs
        if rec_s == 0:
            continue

        val_exc = merged[
            merged['consensus_tier'].isin(VAL_TIERS) &
            (merged['type'].str.upper() == 'EXC')
        ]
        if val_exc.empty:
            continue

        pre_neurons  = set(val_exc['pre'])
        post_neurons = set(val_exc['post'])

        aps_all   = scs.get('aps_all',   {})
        epsps_all = scs.get('epsps_all', {})

        # build EPSP count per post neuron, filling sta_only gaps from STA n_psp
        epsp_counts = {k: len(v) for k, v in epsps_all.items()}
        for _, conn in val_exc[val_exc['consensus_tier'] ==
    'sta_only'].iterrows():
            pre, post = conn['pre'], conn['post']
            if epsp_counts.get(post, 0) == 0:
                result = sta_results.get((pre, post))
                if result is not None and hasattr(result, 'n_psp'):
                    epsp_counts[post] = epsp_counts.get(post, 0) + result.n_psp

        pre_ap_rates    = [len(v) / rec_s for k, v in aps_all.items()
                            if k in pre_neurons and len(v) > 0]
        post_epsp_rates = [epsp_counts.get(k, 0) / rec_s for k in post_neurons
                            if epsp_counts.get(k, 0) > 0]

        if not pre_ap_rates or not post_epsp_rates:
            continue

        fov_rows.append({
            'plate':     R['_plate'],
            'ap_rate':   np.mean(pre_ap_rates),
            'epsp_rate': np.mean(post_epsp_rates),
        })

    df_fov = pd.DataFrame(fov_rows)

    def _iqr_mask(s, k=3.0):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        return (s >= q1 - k * iqr) & (s <= q3 + k * iqr)

    mask = _iqr_mask(df_fov['ap_rate']) & _iqr_mask(df_fov['epsp_rate'])
    df_plot = df_fov[mask]
    n_removed = len(df_fov) - mask.sum()

    fig, ax = plt.subplots(figsize=(4, 3.5))
    x = df_plot['ap_rate'].values
    y = df_plot['epsp_rate'].values

    ax.scatter(x, y, color='#5BA85A', s=20, alpha=0.6)

    if len(x) > 2:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit = np.linspace(x.min(), x.max(), 100)
        ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
        ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                transform=ax.transAxes, fontsize=8)

    ax.set_xlabel('Mean AP rate — EXC pre-synaptic neurons (Hz)', fontsize=8)
    ax.set_ylabel('Mean EPSP rate — EXC post-synaptic neurons (Hz)', fontsize=8)
    ax.set_title(f'EXC pre AP vs post EPSP rate per FOV\n(n={len(df_plot)} FOVs,{n_removed} removed)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'exc_pre_ap_vs_post_epsp_rate.png'),
    dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# Cell XX — 
if not df_conn.empty:
    VAL_TIERS = {'both', 'scs_only', 'sta_only'}

    fov_rows = []
    for R in all_fov_results:
        scs         = R.get('scs1')
        merged      = R.get('merged')
        meta        = R.get('meta', {})
        sta_results = R.get('sta1', {}).get('all_results', {})
        if not scs or merged is None or merged.empty:
            continue

        fs    = int(meta.get('fs', 500) or 500)
        rec_s = (int(meta.get('spont_n_samples', 0) or 0) +
                int(meta.get('stim_n_samples',  0) or 0)) / fs
        if rec_s == 0:
            continue

        val = merged[merged['consensus_tier'].isin(VAL_TIERS)]
        if val.empty:
            continue

        pre_neurons  = set(val['pre'])
        post_neurons = set(val['post'])

        aps_all   = scs.get('aps_all',   {})
        epsps_all = scs.get('epsps_all', {})
        ipsps_all = scs.get('ipsps_all', {})

        # combine SCS EPSP + IPSP counts per post neuron
        psp_counts = {}
        for k in set(epsps_all) | set(ipsps_all):
            psp_counts[k] = len(epsps_all.get(k, [])) + len(ipsps_all.get(k, []))

        # fill sta_only gaps from STA n_psp
        for _, conn in val[val['consensus_tier'] == 'sta_only'].iterrows():
            pre, post = conn['pre'], conn['post']
            if psp_counts.get(post, 0) == 0:
                result = sta_results.get((pre, post))
                if result is not None and hasattr(result, 'n_psp'):
                    psp_counts[post] = psp_counts.get(post, 0) + result.n_psp

        pre_ap_rates  = [len(v) / rec_s for k, v in aps_all.items()
                        if k in pre_neurons and len(v) > 0]
        post_psp_rates = [psp_counts.get(k, 0) / rec_s for k in post_neurons
                        if psp_counts.get(k, 0) > 0]

        if not pre_ap_rates or not post_psp_rates:
            continue

        fov_rows.append({
            'plate':    R['_plate'],
            'ap_rate':  np.mean(pre_ap_rates),
            'psp_rate': np.mean(post_psp_rates),
        })

    df_fov = pd.DataFrame(fov_rows)

    def _iqr_mask(s, k=3.0):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        return (s >= q1 - k * iqr) & (s <= q3 + k * iqr)

    mask = _iqr_mask(df_fov['ap_rate']) & _iqr_mask(df_fov['psp_rate'])
    df_plot = df_fov[mask]
    n_removed = len(df_fov) - mask.sum()

    fig, ax = plt.subplots(figsize=(4, 3.5))
    x = df_plot['ap_rate'].values
    y = df_plot['psp_rate'].values

    ax.scatter(x, y, color='#5B8DB8', s=20, alpha=0.6)

    if len(x) > 2:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit = np.linspace(x.min(), x.max(), 100)
        ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
        ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                transform=ax.transAxes, fontsize=8)

    ax.set_xlabel('Mean AP rate — pre-synaptic neurons (Hz)', fontsize=8)
    ax.set_ylabel('Mean PSP rate — post-synaptic neurons (Hz)', fontsize=8)
    ax.set_title(f'Pre AP vs post PSP rate per FOV\n(n={len(df_plot)} FOVs, {n_removed} removed)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'pre_ap_vs_post_psp_rate.png'), dpi=300,
    bbox_inches='tight')
    plt.close()

In [ ]:
# Cell XX — 
if not df_conn.empty:
    fov_rows = []
    for R in all_fov_results:
        scs  = R.get('scs1')
        meta = R.get('meta', {})
        if not scs:
            continue

        n_sources = int(meta.get('n_sources', 0) or 0)
        if n_sources == 0:
            continue

        fs    = int(meta.get('fs', 500) or 500)
        rec_s = (int(meta.get('spont_n_samples', 0) or 0) +
                int(meta.get('stim_n_samples',  0) or 0)) / fs
        if rec_s == 0:
            continue

        aps_all   = scs.get('aps_all',   {})
        epsps_all = scs.get('epsps_all', {})
        ipsps_all = scs.get('ipsps_all', {})

        active_n = sum(1 for v in aps_all.values() if len(v) > 0)
        if active_n == 0:
            continue

        n_aps  = sum(len(v) for v in aps_all.values())
        n_psps = sum(len(v) for v in epsps_all.values()) + sum(len(v) for v in
    ipsps_all.values())

        fov_rows.append({
            'plate':    R['_plate'],
            'ap_rate':  (n_aps  / active_n)  / rec_s,
            'psp_rate': (n_psps / n_sources) / rec_s,
        })

    df_fov = pd.DataFrame(fov_rows)

    def _iqr_mask(s, k=3.0):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        return (s >= q1 - k * iqr) & (s <= q3 + k * iqr)

    mask = _iqr_mask(df_fov['ap_rate']) & _iqr_mask(df_fov['psp_rate'])
    df_plot = df_fov[mask]
    n_removed = len(df_fov) - mask.sum()

    fig, ax = plt.subplots(figsize=(4, 3.5))
    x = df_plot['ap_rate'].values
    y = df_plot['psp_rate'].values

    ax.scatter(x, y, color='#5B8DB8', s=20, alpha=0.6)

    if len(x) > 2:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit = np.linspace(x.min(), x.max(), 100)
        ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
        ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                transform=ax.transAxes, fontsize=8)

    ax.set_xlabel('Mean AP rate per active neuron (Hz)', fontsize=8)
    ax.set_ylabel('PSP rate per source (Hz)', fontsize=8)
    ax.set_title(f'AP vs PSP rate — all sources\n(n={len(df_plot)} FOVs, {n_removed} removed)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'ap_vs_psp_rate_all_sources.png'),
    dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# Cell XX — 
from utils.data_utils import compute_neuron_snr
if not df_conn.empty:
    SNR_GATE = 3.5

    fov_rows = []
    for R in all_fov_results:
        scs  = R.get('scs1')
        meta = R.get('meta', {})
        if not scs:
            continue

        n_sources = int(meta.get('n_sources', 0) or 0)
        if n_sources == 0:
            continue

        fs    = int(meta.get('fs', 500) or 500)
        rec_s = (int(meta.get('spont_n_samples', 0) or 0) +
                int(meta.get('stim_n_samples',  0) or 0)) / fs
        if rec_s == 0:
            continue

        aps_all    = scs.get('aps_all',    {})
        epsps_all  = scs.get('epsps_all',  {})
        ipsps_all  = scs.get('ipsps_all',  {})
        traces_all = scs.get('traces_all', {})

        mean_snr, above_gate = compute_neuron_snr(traces_all, aps_all,
    snr_gate=SNR_GATE)
        good_sources = {k for k, ok in above_gate.items() if ok}

        active_n = sum(1 for k, v in aps_all.items() if k in good_sources and
    len(v) > 0)
        if active_n == 0:
            continue

        n_aps  = sum(len(v) for k, v in aps_all.items()   if k in good_sources)
        n_psps = sum(len(v) for k, v in epsps_all.items() if k in good_sources) \
                + sum(len(v) for k, v in ipsps_all.items() if k in good_sources)

        n_good = len(good_sources)

        fov_rows.append({
            'plate':    R['_plate'],
            'ap_rate':  (n_aps  / active_n) / rec_s,
            'psp_rate': (n_psps / n_good)   / rec_s,
        })

    df_fov = pd.DataFrame(fov_rows)

    def _iqr_mask(s, k=3.0):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        return (s >= q1 - k * iqr) & (s <= q3 + k * iqr)

    mask = _iqr_mask(df_fov['ap_rate']) & _iqr_mask(df_fov['psp_rate'])
    df_plot = df_fov[mask]
    n_removed = len(df_fov) - mask.sum()

    fig, ax = plt.subplots(figsize=(4, 3.5))
    x = df_plot['ap_rate'].values
    y = df_plot['psp_rate'].values

    ax.scatter(x, y, color='#5B8DB8', s=20, alpha=0.6)

    if len(x) > 2:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit = np.linspace(x.min(), x.max(), 100)
        ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
        ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                transform=ax.transAxes, fontsize=8)

    ax.set_xlabel('Mean AP rate per active neuron (Hz)', fontsize=8)
    ax.set_ylabel('Mean PSP rate per high-SNR source (Hz)', fontsize=8)
    ax.set_title(f'AP vs PSP rate — SNR≥{SNR_GATE} sources\n(n={len(df_plot)} FOVs, {n_removed} removed)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'ap_vs_psp_rate_highsnr.png'), dpi=300,
    bbox_inches='tight')
    plt.close()


In [ ]:
# Cell XX — 
from IPython.display import display

def _effective_window_time(all_pre_aps, post_ap_times, rec_samples, fs,
                            win_start_ms=10, win_end_ms=150,
ahp_blank_ms=100):
    win_s = int(win_start_ms * fs / 1000)
    win_e = int(win_end_ms   * fs / 1000)
    ahp_s = int(ahp_blank_ms * fs / 1000)
    if len(all_pre_aps) == 0:
        return 0.0
    det = sorted((int(t) + win_s, min(int(t) + win_e, rec_samples)) for t in all_pre_aps)
    det = [(s, e) for s, e in det if s < e]
    if not det:
        return 0.0
    merged_det = [list(det[0])]
    for s, e in det[1:]:
        if s <= merged_det[-1][1]:
            merged_det[-1][1] = max(merged_det[-1][1], e)
        else:
            merged_det.append([s, e])
    total_window = sum(e - s for s, e in merged_det)
    if len(post_ap_times) == 0:
        return total_window / fs
    ahp = sorted((int(t), min(int(t) + ahp_s, rec_samples)) for t in post_ap_times)
    merged_ahp = [list(ahp[0])]
    for s, e in ahp[1:]:
        if s <= merged_ahp[-1][1]:
            merged_ahp[-1][1] = max(merged_ahp[-1][1], e)
        else:
            merged_ahp.append([s, e])
    overlap = 0
    ai = 0
    for ds, de in merged_det:
        while ai < len(merged_ahp) and merged_ahp[ai][1] <= ds:
            ai += 1
        j = ai
        while j < len(merged_ahp) and merged_ahp[j][0] < de:
            as_, ae = merged_ahp[j]
            overlap += min(de, ae) - max(ds, as_)
            j += 1
    return (total_window - overlap) / fs

if not df_conn.empty:
    VAL_TIERS = {'both', 'scs_only', 'sta_only'}

    conn_rows   = []
    neuron_rows = []

    for R in all_fov_results:
        scs         = R.get('scs1')
        merged      = R.get('merged')
        meta        = R.get('meta', {})
        sta_results = R.get('sta1', {}).get('all_results', {})
        if not scs or merged is None or merged.empty:
            continue

        fs    = int(meta.get('fs', 500) or 500)
        rec_s = (int(meta.get('spont_n_samples', 0) or 0) +
                int(meta.get('stim_n_samples',  0) or 0)) / fs
        if rec_s == 0:
            continue

        val = merged[merged['consensus_tier'].isin(VAL_TIERS)]
        if val.empty:
            continue

        aps_all   = scs.get('aps_all',   {})
        epsps_all = scs.get('epsps_all', {})
        ipsps_all = scs.get('ipsps_all', {})
        rec_samples = int(rec_s * fs)

        ap_rates = {k: len(v) / rec_s for k, v in aps_all.items()}

        psp_counts = {}
        for k in set(epsps_all) | set(ipsps_all):
            psp_counts[k] = len(epsps_all.get(k, [])) + len(ipsps_all.get(k,[]))

        for _, conn in val[val['consensus_tier'] == 'sta_only'].iterrows():
            pre, post = conn['pre'], conn['post']
            if psp_counts.get(post, 0) == 0:
                result = sta_results.get((pre, post))
                if result is not None and hasattr(result, 'n_psp'):
                    psp_counts[post] = psp_counts.get(post, 0) + result.n_psp

        # cache effective window time per post neuron
        _eff_time_cache = {}
        def _get_eff_time(post):
            if post not in _eff_time_cache:
                pre_aps = np.concatenate([v for k, v in aps_all.items()
                                        if k != post and len(v) > 0]) if aps_all else np.array([])
                _eff_time_cache[post] = _effective_window_time(
                    pre_aps, aps_all.get(post, []), rec_samples, fs)
            return _eff_time_cache[post]

        for _, conn in val.iterrows():
            pre  = conn['pre']
            post = conn['post']
            typ  = conn['type'].upper()
            if pre not in ap_rates or ap_rates[pre] == 0:
                continue
            if psp_counts.get(post, 0) == 0:
                continue
            eff_time = _get_eff_time(post)
            if eff_time <= 0:
                continue
            conn_rows.append({
                'plate':         R['_plate'],
                'type':          typ,
                'pre_ap_rate':   ap_rates[pre],
                'post_psp_rate': psp_counts[post] / eff_time,
            })

        pre_to_posts = {}
        for _, conn in val.iterrows():
            pre  = conn['pre']
            post = conn['post']
            typ  = conn['type'].upper()
            if pre not in ap_rates or ap_rates[pre] == 0:
                continue
            if psp_counts.get(post, 0) == 0:
                continue
            eff_time = _get_eff_time(post)
            if eff_time <= 0:
                continue
            pre_to_posts.setdefault((pre, typ), []).append(psp_counts[post] / eff_time)

        for (pre, typ), post_rates in pre_to_posts.items():
            neuron_rows.append({
                'plate':              R['_plate'],
                'type':               typ,
                'pre_ap_rate':        ap_rates[pre],
                'mean_post_psp_rate': np.mean(post_rates),
                'n_partners':         len(post_rates),
            })

    df_ap_psp   = pd.DataFrame(conn_rows)
    df_neuron = pd.DataFrame(neuron_rows)

    def _iqr_mask(s, k=3.0):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        return (s >= q1 - k * iqr) & (s <= q3 + k * iqr)

    type_colors = {'EXC': '#5BA85A', 'INH': '#E07A2F'}

    # Figure 1 — per connection overlaid + per pre neuron
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))

    ax = axes[0]
    for i, (typ, color) in enumerate(type_colors.items()):
        df_t = df_ap_psp[df_ap_psp['type'] == typ]
        mask = _iqr_mask(df_t['pre_ap_rate']) & _iqr_mask(df_t['post_psp_rate'])
        df_p = df_t[mask]
        ax.scatter(df_p['pre_ap_rate'], df_p['post_psp_rate'],
                    color=color, s=10, alpha=0.4, label=typ)
        if len(df_p) > 2:
            x, y = df_p['pre_ap_rate'].values, df_p['post_psp_rate'].values
            slope, intercept, r, p, _ = stats.linregress(x, y)
            xfit = np.linspace(x.min(), x.max(), 100)
            ax.plot(xfit, slope * xfit + intercept, color=color, lw=1.5)
            ax.text(0.05, 0.92 - i * 0.08, f'{typ} R²={r**2:.2f} p={p:.2e}',
                    transform=ax.transAxes, fontsize=7, color=color)
    ax.set_xlabel('Pre-synaptic AP rate (Hz)', fontsize=8)
    ax.set_ylabel('Post-synaptic PSP rate (Hz)', fontsize=8)
    ax.set_title('Per connection', fontsize=9)
    ax.legend(frameon=False, fontsize=7)
    ax.spines[['top', 'right']].set_visible(False)

    ax = axes[1]
    for i, (typ, color) in enumerate(type_colors.items()):
        df_t = df_neuron[df_neuron['type'] == typ]
        mask = _iqr_mask(df_t['pre_ap_rate']) & _iqr_mask(df_t['mean_post_psp_rate'])
        df_p = df_t[mask]
        ax.scatter(df_p['pre_ap_rate'], df_p['mean_post_psp_rate'],
                    color=color, s=10, alpha=0.4, label=typ)
        if len(df_p) > 2:
            x, y = df_p['pre_ap_rate'].values, df_p['mean_post_psp_rate'].values
            slope, intercept, r, p, _ = stats.linregress(x, y)
            xfit = np.linspace(x.min(), x.max(), 100)
            ax.plot(xfit, slope * xfit + intercept, color=color, lw=1.5)
            ax.text(0.05, 0.92 - i * 0.08, f'{typ} R²={r**2:.2f} p={p:.2e}',
                    transform=ax.transAxes, fontsize=7, color=color)
    ax.set_xlabel('Pre-synaptic AP rate (Hz)', fontsize=8)
    ax.set_ylabel('Mean post-synaptic PSP rate (Hz)', fontsize=8)
    ax.set_title('Per pre neuron\n(mean over post-synaptic partners)', fontsize=9)
    ax.legend(frameon=False, fontsize=7)
    ax.spines[['top', 'right']].set_visible(False)

    fig.suptitle('Pre AP rate vs post PSP rate — validated connections',
    fontsize=10)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, 'pre_ap_vs_post_psp_rate.png'), dpi=300,
    bbox_inches='tight')
    plt.close()
 

    # Figure 2 — per connection, EXC and INH split
    fig2, axes2 = plt.subplots(1, 2, figsize=(8, 4))
    for ax2, typ in zip(axes2, ['EXC', 'INH']):
        color = type_colors[typ]
        df_t  = df_ap_psp[df_ap_psp['type'] == typ]
        mask  = _iqr_mask(df_t['pre_ap_rate']) & _iqr_mask(df_t['post_psp_rate'])
        df_p  = df_t[mask]
        ax2.scatter(df_p['pre_ap_rate'], df_p['post_psp_rate'],
                    color=color, s=10, alpha=0.4)
        if len(df_p) > 2:
            x, y = df_p['pre_ap_rate'].values, df_p['post_psp_rate'].values
            slope, intercept, r, p, _ = stats.linregress(x, y)
            xfit = np.linspace(x.min(), x.max(), 100)
            ax2.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
            ax2.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                    transform=ax2.transAxes, fontsize=8)
        ax2.set_xlabel('Pre-synaptic AP rate (Hz)', fontsize=8)
        ax2.set_ylabel('Post-synaptic PSP rate (Hz)', fontsize=8)
        ax2.set_title(f'Per connection — {typ} (n={len(df_p)})', fontsize=9)
        ax2.spines[['top', 'right']].set_visible(False)

    fig2.suptitle('Pre AP rate vs post PSP rate — EXC / INH split', fontsize=10)
    fig2.tight_layout()
    fig2.savefig(os.path.join(OUTPUT_DIR,
    'pre_ap_vs_post_psp_per_conn_split.png'), dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# Cell XX — 
# # Total APs vs PSPs per well
if not df_conn.empty:
    well_rows = []
    for R in all_fov_results:
        scs  = R.get('scs1')
        if not scs:
            continue

        fov   = R['fov']
        plate = R['_plate']
        well  = fov_well_map.get((fov, plate), 'unknown')

        aps_all   = scs.get('aps_all',   {})
        epsps_all = scs.get('epsps_all', {})
        ipsps_all = scs.get('ipsps_all', {})

        n_aps  = sum(len(v) for v in aps_all.values())
        n_psps = sum(len(v) for v in epsps_all.values()) + sum(len(v) for v in
    ipsps_all.values())

        well_rows.append({'plate': plate, 'well': well, 'n_aps': n_aps, 'n_psps':
    n_psps})

    df_well = (pd.DataFrame(well_rows)
                .groupby(['plate', 'well'])[['n_aps', 'n_psps']]
                .sum()
                .reset_index())

    fig, ax = plt.subplots(figsize=(4, 3.5))
    x = df_well['n_aps'].values
    y = df_well['n_psps'].values

    ax.scatter(x, y, color='#5B8DB8', s=20, alpha=0.6)

    if len(x) > 2:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit = np.linspace(x.min(), x.max(), 100)
        ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
        ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                transform=ax.transAxes, fontsize=8)

    ax.set_xlabel('Total APs per well', fontsize=8)
    ax.set_ylabel('Total PSPs per well', fontsize=8)
    ax.set_title(f'AP vs PSP counts per well (n={len(df_well)} wells)',fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'ap_vs_psp_per_well.png'), dpi=300,
    bbox_inches='tight')
    plt.close()

In [ ]:
# Cell XX — 
# # Number of sources per FOV
if not df_conn.empty:
    fov_rows = []
    for R in all_fov_results:
        scs  = R.get('scs1')
        meta = R.get('meta', {})
        if not scs:
            continue

        n_sources = int(meta.get('n_sources', 0) or 0)
        if n_sources == 0:
            continue

        epsps_all = scs.get('epsps_all', {})
        ipsps_all = scs.get('ipsps_all', {})

        n_psps = sum(len(v) for v in epsps_all.values()) + sum(len(v) for v in
    ipsps_all.values())

        fov_rows.append({'plate': R['_plate'], 'n_sources': n_sources, 'n_psps':
    n_psps})

    df_fov = pd.DataFrame(fov_rows)

    def _iqr_mask(s, k=3.0):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        return (s >= q1 - k * iqr) & (s <= q3 + k * iqr)

    mask = _iqr_mask(df_fov['n_sources']) & _iqr_mask(df_fov['n_psps'])
    df_plot_sc = df_fov[mask]
    n_removed = len(df_fov) - mask.sum()

    fig, ax = plt.subplots(figsize=(4, 3.5))
    x = df_plot_sc['n_sources'].values
    y = df_plot_sc['n_psps'].values

    ax.scatter(x, y, color='#5B8DB8', s=20, alpha=0.6)

    if len(x) > 2:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit = np.linspace(x.min(), x.max(), 100)
        ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
        ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                transform=ax.transAxes, fontsize=8)

    ax.set_xlabel('Number of sources per FOV', fontsize=8)
    ax.set_ylabel('Total PSPs per FOV', fontsize=8)
    ax.set_title(f'Sources vs total PSP count\n(n={len(df_plot_sc)} FOVs, {n_removed} removed)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'sources_vs_total_psps.png'), dpi=300,
    bbox_inches='tight')
    plt.close()

In [ ]:
# Cell XX — 
# Stimulated neuron validated pairs only
if not df_conn.empty:
    VAL_TIERS = {'both', 'scs_only', 'sta_only'}

    neuron_rows = []
    for R in all_fov_results:
        merged      = R.get('merged')
        sta         = R.get('sta1', {})
        if merged is None or merged.empty or not sta:
            continue

        val = merged[merged['consensus_tier'].isin(VAL_TIERS)]
        if val.empty:
            continue

        sta_results  = sta.get('all_results', {})
        aps_for_sta  = sta.get('aps_for_sta', {})

        # group validated connections by pre neuron
        for pre, grp in val.groupby('pre'):
            n_stim_aps = len(aps_for_sta.get(pre, []))
            if n_stim_aps == 0:
                continue

            n_psp_total      = 0
            n_ap_evoked_total = 0
            for _, conn in grp.iterrows():
                post   = conn['post']
                result = sta_results.get((pre, post))
                if result is None:
                    continue
                n_psp_total       += getattr(result, 'n_psp',       0) or 0
                n_ap_evoked_total += getattr(result, 'n_ap_evoked', 0) or 0

            if n_psp_total == 0 and n_ap_evoked_total == 0:
                continue

            neuron_rows.append({
                'plate':          R['_plate'],
                'n_stim_aps':     n_stim_aps,
                'n_psp':          n_psp_total,
                'n_ap_evoked':    n_ap_evoked_total,
                'n_partners':     len(grp),
            })

    df_neuron = pd.DataFrame(neuron_rows)

    def _iqr_mask(s, k=3.0):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        return (s >= q1 - k * iqr) & (s <= q3 + k * iqr)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))

    for ax, ycol, ylabel, color in [
        (axes[0], 'n_psp',       'Total subthreshold PSPs in partners', "#043E70"),
        (axes[1], 'n_ap_evoked', 'Total evoked APs in partners', "#07030000"),
    ]:
        mask = _iqr_mask(df_neuron['n_stim_aps']) & _iqr_mask(df_neuron[ycol])
        df_p = df_neuron[mask]
        n_removed = len(df_neuron) - mask.sum()

        x = df_p['n_stim_aps'].values
        y = df_p[ycol].values
        ax.scatter(x, y, color=color, s=15, alpha=0.5)

        if len(x) > 2:
            slope, intercept, r, p, _ = stats.linregress(x, y)
            xfit = np.linspace(x.min(), x.max(), 100)
            ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
            ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                    transform=ax.transAxes, fontsize=8)

        ax.set_xlabel('APs fired during stimulation (pre neuron)', fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.set_title(f'n={len(df_p)} neurons, {n_removed} removed', fontsize=8)
        ax.spines[['top', 'right']].set_visible(False)

    fig.suptitle('Stimulation-evoked pre APs vs post-synaptic responses',
    fontsize=10)
    fig.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'stim_aps_vs_partner_responses.png'),
    dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# Cell XX — 
if not df_conn.empty:
    fov_rows = []
    for R in all_fov_results:
        scs  = R.get('scs1')
        if not scs:
            continue

        aps_all   = scs.get('aps_all',   {})
        epsps_all = scs.get('epsps_all', {})
        ipsps_all = scs.get('ipsps_all', {})

        n_active = sum(1 for v in aps_all.values() if len(v) > 0)
        if n_active == 0:
            continue

        n_psps = sum(len(v) for v in epsps_all.values()) + sum(len(v) for v in
    ipsps_all.values())

        fov_rows.append({'plate': R['_plate'], 'n_active': n_active, 'n_psps':
    n_psps})

    df_fov = pd.DataFrame(fov_rows)

    def _iqr_mask(s, k=3.0):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        return (s >= q1 - k * iqr) & (s <= q3 + k * iqr)

    mask = _iqr_mask(df_fov['n_active']) & _iqr_mask(df_fov['n_psps'])
    df_plot_sc = df_fov[mask]
    n_removed = len(df_fov) - mask.sum()

    fig, ax = plt.subplots(figsize=(4, 3.5))
    x = df_plot_sc['n_active'].values
    y = df_plot_sc['n_psps'].values

    ax.scatter(x, y, color='#5B8DB8', s=20, alpha=0.6)

    if len(x) > 2:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit = np.linspace(x.min(), x.max(), 100)
        ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
        ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                transform=ax.transAxes, fontsize=8)

    ax.set_xlabel('Number of active neurons per FOV', fontsize=8)
    ax.set_ylabel('Total PSPs per FOV', fontsize=8)
    ax.set_title(f'Active neurons vs total PSP count\n(n={len(df_plot_sc)} FOVs,{n_removed} removed)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'active_neurons_vs_total_psps.png'),
    dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# Cell XX — 
if not df_conn.empty:
    fov_rows = []
    for R in all_fov_results:
        scs  = R.get('scs1')
        if not scs:
            continue

        aps_all   = scs.get('aps_all',   {})
        epsps_all = scs.get('epsps_all', {})
        ipsps_all = scs.get('ipsps_all', {})

        active = {k for k, v in aps_all.items() if len(v) > 0}
        if not active:
            continue

        n_active = len(active)
        n_aps    = sum(len(aps_all[k]) for k in active)
        n_psps   = sum(len(v) for v in epsps_all.values()) + sum(len(v) for v in
    ipsps_all.values())

        fov_rows.append({'plate': R['_plate'], 'n_aps': n_aps, 'n_psps': n_psps /
    n_active})

    df_fov = pd.DataFrame(fov_rows)

    def _iqr_mask(s, k=3.0):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        return (s >= q1 - k * iqr) & (s <= q3 + k * iqr)

    mask = _iqr_mask(df_fov['n_aps']) & _iqr_mask(df_fov['n_psps'])
    df_plot_sc = df_fov[mask]
    n_removed = len(df_fov) - mask.sum()

    fig, ax = plt.subplots(figsize=(4, 3.5))
    x = df_plot_sc['n_aps'].values
    y = df_plot_sc['n_psps'].values

    ax.scatter(x, y, color='#5B8DB8', s=20, alpha=0.6)

    if len(x) > 2:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit = np.linspace(x.min(), x.max(), 100)
        ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
        ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                transform=ax.transAxes, fontsize=8)

    ax.set_xlabel('Total APs in active neurons', fontsize=8)
    ax.set_ylabel('Total PSPs per active neuron', fontsize=8)
    ax.set_title(f'APs vs PSPs — all detected PSPs\n(n={len(df_plot_sc)} FOVs, {n_removed} removed)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'aps_vs_psps_all_detected.png'), dpi=300,
    bbox_inches='tight')
    plt.close()

In [ ]:
# Reload utils.visualization during pipeline development
importlib.reload(utils.visualization)
from utils.visualization import (
    build_condition_feature_df, filter_fovs,
    build_variance_table, _cohens_d,
    plot_condition_heatmap, plot_plate_heatmap,
    plot_well_feature_heatmap, plot_plate_condition_heatmap,
    plot_network_fingerprint, plot_circuit_composition,
    plot_fanin_fanout, plot_feature_boxplots, pool_conditions,
    plot_pharmacology_dissociation, plot_synaptic_coupling,
)
importlib.reload(utils.scs_utils)
from utils.scs_utils import (_psp_props_one, compute_psp_props_fov)

In [ ]:
# Cell XX — 
plot_condition_heatmap(df_features, vmax=4, output_path=os.path.join(OUTPUT_DIR,'plate_condition_heatmap.png')) # heatmap of features
plot_network_fingerprint(df_features, output_path=os.path.join(OUTPUT_DIR,'network_fingerprint.png')) # radial plots
plot_circuit_composition(df_features, output_path=os.path.join(OUTPUT_DIR,'circuit_composition.png')) # 
plot_fanin_fanout(df_features, output_path=os.path.join(OUTPUT_DIR, 'fanin_fanout.png')) # convergence vs. divergence

In [ ]:
# Cell XX — 
#plot_plate_heatmap(df_features, output_path=os.path.join(OUTPUT_DIR, 'plate_heatmap.png'))
#plot_plate_condition_heatmap(df_features, vmax=4, output_path=os.path.join(OUTPUT_DIR,'plate_condition_heatmap.png'))
#plot_well_feature_heatmap(df_features, cmap='Purples', output_path=os.path.join(OUTPUT_DIR,'well_feature_heatmap.png'))

In [ ]:
# in progress - check function
plot_pharmacology_dissociation(df_features, output_path=os.path.join(OUTPUT_DIR, 'pharmacology_dissociation.png'))
plot_synaptic_coupling(df_features,output_path=os.path.join(OUTPUT_DIR, 'synaptic_coupling.png'))

In [ ]:
# Cell XX — 
plot_feature_boxplots(df_features, output_path=os.path.join(OUTPUT_DIR, 'boxplots.png'))

In [ ]:
# Cell XX — 
# CoV of key metrics per plate — EXC / INH
if not df_conn.empty:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    plates = list(PLATE_DIRS.keys())
    x = np.arange(len(plates))
    width = 0.3

    def _cov(s):
        m = s.mean()
        return s.std(ddof=1) / m * 100 if m and not np.isnan(m) else np.nan

    metrics = [
        ('Validation rate (%)',  vr,          'rate',               'rate'),
        ('Connection count',     df_features, 'conn_n_exc',
'conn_n_inh'),
        ('AP count',             df_features, 'ap_mean_count_exc',
'ap_mean_count_inh'),
    ]

    for ax, (title, df, exc_col, inh_col) in zip(axes, metrics):
        for j, (typ, color, col) in enumerate([('EXC', '#5BA85A', exc_col),
                                                ('INH', '#E07A2F', inh_col)]):
            if df is vr:
                covs = [_cov(df.loc[(df['plate'] == p) & (df['type'] == typ),
col])
                        for p in plates]
            else:
                covs = [_cov(df.loc[df['plate'] == p, col].dropna())
                        for p in plates]
            offset = (j - 0.5) * width
            ax.bar(x + offset, covs, width=width, color=color, label=typ)

        ax.set_xticks(x)
        ax.set_xticklabels(plates, rotation=30, ha='right', fontsize=8)
        ax.set_ylabel('CoV (%)')
        ax.set_title(title)
        ax.legend(frameon=False)
        ax.spines[['top', 'right']].set_visible(False)

    plt.suptitle('Within-plate variability (CoV) across plates', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'cov_metrics.png'), dpi=300,
  bbox_inches='tight')
    plt.close()

In [ ]:
# Cell XX — Compute stability metrics per FOV
if not df_conn.empty:
    PIXEL_SIZE_UM = 8.0
    VAL_TIERS = {'both', 'scs_only', 'sta_only'}

    stab_rows = []
    for R in all_fov_results:
        fov        = R['fov']
        plate      = R['_plate']
        meta       = R.get('meta', {})
        scs        = R.get('scs1')
        fov_merged = R.get('merged')

        fs    = int(meta.get('fs', 500) or 500)
        rec_s = (int(meta.get('spont_n_samples', 0) or 0) +
                int(meta.get('stim_n_samples',  0) or 0)) / fs

        if scs:
            epsp_n = sum(len(v) for v in scs.get('epsps_all', {}).values())
            ipsp_n = sum(len(v) for v in scs.get('ipsps_all', {}).values())
        else:
            epsp_n = ipsp_n = 0
        psp_rate_exc = epsp_n / rec_s if rec_s else np.nan
        psp_rate_inh = ipsp_n / rec_s if rec_s else np.nan

        rows_arr = np.array(meta.get('source_centroid_row', []), dtype=float)
        cols_arr = np.array(meta.get('source_centroid_col', []), dtype=float)
        mean_dist = np.nan
        if fov_merged is not None and not fov_merged.empty and len(rows_arr):
            val = fov_merged[fov_merged['consensus_tier'].isin(VAL_TIERS)]
            dists = []
            for _, conn in val.iterrows():
                try:
                    pre_idx  = int(str(conn['pre']).lstrip('T'))  - 1
                    post_idx = int(str(conn['post']).lstrip('T')) - 1
                    if 0 <= pre_idx < len(rows_arr) and 0 <= post_idx < len(rows_arr):
                        d = np.sqrt((rows_arr[pre_idx] - rows_arr[post_idx])**2 +
                                    (cols_arr[pre_idx] - cols_arr[post_idx])**2)
                        dists.append(d * PIXEL_SIZE_UM)
                except (ValueError, TypeError):
                    continue
            mean_dist = float(np.mean(dists)) if dists else np.nan

        stab_rows.append({
            'fov':          fov,
            'plate':        plate,
            'psp_rate_exc': psp_rate_exc,
            'psp_rate_inh': psp_rate_inh,
            'mean_dist_um': mean_dist,
        })

    df_stab = pd.DataFrame(stab_rows).merge(
        df_features[['fov', 'plate', 'neuron_n_total',
                    'ap_mean_rate_exc', 'ap_mean_rate_inh', 'conn_n_total']], on=['fov', 'plate'], how='left'
    )
    
    df_features = df_features.merge(
        df_stab[['fov', 'plate', 'mean_dist_um', 'psp_rate_exc', 'psp_rate_inh']],
        on=['fov', 'plate'], how='left'
    )

# Cell XX — Plot stability metrics across plates
plates = list(PLATE_DIRS.keys())
if not df_conn.empty:
    width = 0.15

    panels = [
        ('neuron_n_total',   None,               'Active neurons',     'Active neurons',  0.2),
        ('conn_n_total',     None,               'Total connections', 'Connections',     0.2),
        ('mean_dist_um',     None,               'Distance (µm)', 'Distance (µm)',   0.2),
        ('ap_mean_rate_exc', 'ap_mean_rate_inh', 'Spike frequency (Hz)', 'Spike freq (Hz)', 0.4),
        ('psp_rate_exc',     'psp_rate_inh',     'PSP rate (Hz)',      'PSP rate (Hz)',   0.4),
    ]

    fig, axes = plt.subplots(1, 5, figsize=(16, 3),
                            gridspec_kw={'width_ratios': [1, 1, 1, 1.8,
1.8]})

    for ax, (exc_col, inh_col, title, ylabel, x_scale) in zip(axes, panels):
        for i, plate in enumerate(plates):
            grp = df_stab[df_stab['plate'] == plate]
            x   = i * x_scale

            if inh_col is None:
                vals = grp[exc_col].dropna()
                n    = len(vals)
                mean = vals.mean() if n else 0
                sem  = vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0
                ax.bar(x, mean, yerr=sem, color='gray', width=width,
                        capsize=5, error_kw=dict(elinewidth=1.5))
                ax.scatter([x] * n, vals, color='k', s=20, zorder=5, alpha=0.6)
                ax.text(x, 0, f'{n} FOVs', ha='center', va='bottom', fontsize=6)
            else:
                for j, (col, color, typ) in enumerate([
                    (exc_col, '#5BA85A', 'EXC'), (inh_col, '#E07A2F', 'INH')
                ]):
                    vals = grp[col].dropna()
                    n    = len(vals)
                    mean = vals.mean() if n else 0
                    sem  = vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0
                    xj   = x + (j - 0.5) * (width + 0.05)
                    ax.bar(xj, mean, yerr=sem, color=color, width=width,
                            capsize=5, error_kw=dict(elinewidth=1.5),
                            label=typ if i == 0 else '')
                    ax.scatter([xj] * n, vals, color='k', s=20, zorder=5, alpha=0.6)
                ax.text(x, 0, f'{n} FOVs', ha='center', va='bottom', fontsize=6)

        ax.set_xticks([i * x_scale for i in range(len(plates))])
        ax.set_xticklabels(plates, ha='center', fontsize=8)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.spines[['top', 'right']].set_visible(False)
        if inh_col:
            ax.legend(frameon=False, fontsize=7)

    plt.suptitle('Network stability across plates', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'network_stability.png'), dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Cell XX — 
# # Variance table
styled = build_variance_table(df_features, output_path=os.path.join(OUTPUT_DIR,'variance_table.csv'))
display(styled)

In [ ]:
importlib.reload(utils.visualization)
from utils.visualization import plot_power_analysis
n_results = plot_power_analysis(
    df_features,
    effect_sizes=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    output_path=os.path.join(OUTPUT_DIR, 'power_analysis.png'),
)

# Generates information about the pipeline

In [ ]:
raise StopIteration("Stopping here intentionally")

In [ ]:
pipeline_params = pd.DataFrame([
    ('Drug condition',            'metadata CSV',               'drug1',
                'No Drug, GABAzine, CNQX'),
    ('Well ID',                   'metadata CSV',               'wellId',
                'A01, B03, F06'),
    ('Stimulation protocol',      'meta (pickle)',
'stim_step_name',             'SingleCellMaskSwitchingComb'),
    ('Recording segments',        'meta (pickle)',
'spont/stim_n_samples',       '6907 / 32320 frames'),
    ('Stimulus frames per source','meta (pickle)',
'stim_frames_per_source',     'per-neuron frame indices'),
    ('Number of sources',         'meta (pickle)',              'n_sources',
                '239'),
    ('Sampling rate',             'meta (pickle)',              'fs',
                '500 Hz'),
    ('Compatible pipelines',      'Quiver platform',            '—',
                'Excitability, Spontaneous'),
], columns=['Parameter', 'Source', 'Key', 'Example'])

styled_params = (pipeline_params.style
                .hide(axis='index')
                .set_caption('Pipeline flexibility — supported metadata parameters')
                .set_table_styles([{'selector': 'th, td', 'props':
[('font-size', '9px')]}]))
display(styled_params)

In [ ]:
edge_cases = pd.DataFrame([
    ('Near-coincident spike timing',  'Automatic',  'PSP detection window parameters filter overlapping events'),
    ('Small amplitude PSPs',          'Automatic',  'Detrending + spike-triggered averaging recovers low-SNR PSPs'),
    ('Silent/unstimulated sources',   'Automatic',  'Flagged via stim_mask_sources; excluded from connection mapping'),
    ('Failed FOVs',                   'Automatic',  'Try/except per FOV; failed FOVs logged and skipped without stopping batch'),
    ('Duplicate neurons',             'Automatic',  'find_duplicate_neurons removes redundant sources before analysis'),
    ('Missing metadata',              'Automatic',  'Pipeline runs without metadata; condition defaults to No Drug'),
], columns=['Edge Case', 'Handling', 'Detail'])

styled_edge = (edge_cases.style
                .hide(axis='index')
                .set_caption('Pipeline adaptability — edge case handling')
                .set_table_styles([{'selector': 'th, td', 'props':
[('font-size', '9px')]}]))
display(styled_edge)

In [ ]:
version_control = pd.DataFrame([
      ('Source code',          'GitHub',       'All pipeline modules under version control'),
      ('Analysis outputs',     'GitHub',       'Output filenames include pipeline version tag'),
      ('Validation split',     'Fixed seed',   'Random seed fixed before train/test split — reproducible across runs'),
      ('Notebook versioning',  'GitHub',       'Analysis notebooks committed alongside pipeline code'),
      ('Dependency tracking',  'requirements / conda env', 'Package versions pinned in environment file'), ], columns=['Component', 'Mechanism', 'Detail'])

styled_vc = (version_control.style
            .hide(axis='index')
            .set_caption('Reproducibility — version control and analysis traceability')
            .set_table_styles([{'selector': 'th, td', 'props': [('font-size',
'9px')]}]))
display(styled_vc)